# Fluvius Phase 1: free-GPU timing pilot

Run one **1,000-update development pilot** on a Colab or Kaggle GPU, then export
its complete record. This tests whether a GPU is useful for the current small
network; it is **not a completed seed study or evidence of publication readiness**.
The architecture and global-standardization configuration come from the V3
normalization screen. The hardware backend and total update budget differ from
that CPU screen and are recorded explicitly.

**Before running:** enable a GPU in the notebook provider, enable internet for
package installation, and upload the supplied `inner24-transfer.zip`. In Kaggle,
attach it as a private dataset if appropriate. Enter its actual path below. The
archive contains only the fixed inner TRAIN/VAL split; it contains no original
benchmark VAL or TEST. No Azure resources, paid services, Git pushes or Drive
mounts are created here. Free GPU availability and quotas depend on your account.

**Validation status:** notebook structure and Python cell syntax were checked on
the CPU development machine. CUDA execution and provider-specific presentation
remain unverified. Run all cells on the selected GPU and preserve the resulting
archive before relying on its output.

## 1. Choose the input and pilot

Set `DATA_ARCHIVE` to the uploaded archive. Keep the default budget for the first
hardware trial. Use a new attempt for each run; failed attempts are retained.

In [ ]:
from pathlib import Path
import datetime as dt
import hashlib
import json
import os
import subprocess
import sys
import time
import zipfile

# Colab example: /content/inner24-transfer.zip
# Kaggle example: /kaggle/input/fluvius-inner24/inner24-transfer.zip
DATA_ARCHIVE = Path('/content/inner24-transfer.zip')
PILOT_STEPS = 1000
RUN_CONTINUATION = False  # Optional completed-segment continuation, later in this notebook.
CONTINUATION_TOTAL_STEPS = 6000
REPOSITORY_URL = 'https://github.com/MahmoudAsadi97/cGAN.git'

if Path('/kaggle/working').is_dir():
    WORKSPACE = Path('/kaggle/working')
elif Path('/content').is_dir():
    WORKSPACE = Path('/content')
else:
    WORKSPACE = Path.cwd()
PROJECT = WORKSPACE / 'cGAN'
ATTEMPT = dt.datetime.now(dt.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
if sys.version_info[:2] < (3, 11):
    raise RuntimeError('Python 3.11 or newer is required.')
if PILOT_STEPS != 1000:
    raise ValueError('This pilot declares 1000 updates. Use a separate plan for another budget.')
if not DATA_ARCHIVE.is_file():
    raise FileNotFoundError(f'Upload inner24-transfer.zip and set DATA_ARCHIVE: {DATA_ARCHIVE}')
print({'workspace': str(WORKSPACE), 'attempt': ATTEMPT, 'pilot_steps': PILOT_STEPS})

## 2. Clone V3 and check the provider GPU

An existing checkout is verified and preserved. This cell does not pull, reset,
or switch another project. The exact commit is recorded for the attempt.

In [ ]:
def command(args, *, cwd=None, env=None, capture=False):
    result = subprocess.run([str(arg) for arg in args], cwd=cwd, env=env,
                            text=True, capture_output=capture, check=True)
    return result.stdout.strip() if capture else result

if not PROJECT.exists():
    command(['git', 'clone', '--single-branch', '--branch', 'V3', REPOSITORY_URL, PROJECT])
if not (PROJECT / '.git').exists():
    raise RuntimeError(f'Existing path is not this project: {PROJECT}')
origin = command(['git', 'remote', 'get-url', 'origin'], cwd=PROJECT, capture=True)
branch = command(['git', 'branch', '--show-current'], cwd=PROJECT, capture=True)
if origin.removesuffix('.git') != REPOSITORY_URL.removesuffix('.git') or branch != 'V3':
    raise RuntimeError('Expected MahmoudAsadi97/cGAN on branch V3; existing checkout left unchanged.')
if command(['git', 'status', '--porcelain', '--untracked-files=no'], cwd=PROJECT, capture=True):
    raise RuntimeError('Tracked project files have local edits; preserve them and use a clean V3 checkout.')
SOURCE_COMMIT = command(['git', 'rev-parse', 'HEAD'], cwd=PROJECT, capture=True)
ENV = os.environ.copy()
ENV.update(OMP_NUM_THREADS='1', MKL_NUM_THREADS='1', OPENBLAS_NUM_THREADS='1',
           CUBLAS_WORKSPACE_CONFIG=':4096:8', PIP_DISABLE_PIP_VERSION_CHECK='1')
provider_check = """
import json, sys, torch
version = tuple(int(part) for part in torch.__version__.split('+')[0].split('.')[:2])
assert (2, 8) <= version < (3, 0), 'Provider PyTorch must be >=2.8,<3; use a compatible GPU runtime.'
assert torch.cuda.is_available(), 'Enable an available GPU runtime before continuing.'
print(json.dumps({'python': sys.version.split()[0], 'torch': str(torch.__version__),
                  'cuda': torch.version.cuda, 'gpu': torch.cuda.get_device_name(0)}))
"""
PROVIDER = json.loads(command([sys.executable, '-c', provider_check], env=ENV, capture=True))
print({'source_commit': SOURCE_COMMIT, **PROVIDER})

## 3. Prepare an isolated project environment

Reuse the provider's CUDA-enabled PyTorch and install the numerical dependencies
inside this checkout. The CPU requirements file is deliberately not used here.
The provider's system packages are not modified. Exact installed versions are
saved with the run. A compatible provider PyTorch installation is required;
this notebook does not replace its GPU driver or CUDA build.

In [ ]:
ENV_ROOT = PROJECT / '.cache' / 'gpu-notebook' / ATTEMPT
VENV = ENV_ROOT / 'venv'
ENV_ROOT.mkdir(parents=True, exist_ok=False)
(ENV_ROOT / 'tmp').mkdir()
ENV['TMPDIR'] = str(ENV_ROOT / 'tmp')
ENV['PIP_CACHE_DIR'] = str(PROJECT / '.cache' / 'pip-gpu-notebook')
command([sys.executable, '-m', 'venv', '--system-site-packages', VENV], env=ENV)
PYTHON = VENV / 'bin' / 'python'
command([PYTHON, '-m', 'pip', 'install', '--quiet',
         'numpy==2.2.6', 'pandas==2.2.3', 'scipy==1.15.3',
         'setuptools==80.9.0', 'wheel==0.45.1'], env=ENV)
command([PYTHON, '-m', 'pip', 'install', '--quiet', '--no-deps',
         '--no-build-isolation', '--editable', PROJECT], env=ENV)
compatibility_check = """
import inspect, json, numpy, pandas, scipy, torch
from fluvius_cgan.development import _config, train_probe
assert numpy.__version__ == '2.2.6'
assert pandas.__version__ == '2.2.3'
assert scipy.__version__ == '1.15.3'
assert torch.cuda.is_available()
assert _config({'device': 'cuda'})['device'] == 'cuda'
assert 'resume' in inspect.signature(train_probe).parameters
print(json.dumps({'numpy': numpy.__version__, 'pandas': pandas.__version__,
                  'scipy': scipy.__version__, 'torch': str(torch.__version__),
                  'cuda': torch.version.cuda, 'gpu': torch.cuda.get_device_name(0)}))
"""
RUNTIME = json.loads(command([PYTHON, '-c', compatibility_check], cwd=PROJECT, env=ENV, capture=True))
print(RUNTIME)

## 4. Verify and unpack the fixed development split

Every archive member must match the committed normalization plan byte for byte.
Extraction accepts only the four named files, so archive paths cannot write
outside the new data directory. Do not bypass a mismatch by editing the plan.
The source paths inside metadata describe the original preparation machine;
the current location is recorded separately below.

In [ ]:
def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

PLAN_PATH = PROJECT / 'reports' / 'phase1-development' / 'NORMALIZATION_SCREEN_PLAN.json'
PLAN = json.loads(PLAN_PATH.read_text())
EXPECTED = PLAN['data_hashes']
if set(EXPECTED) != {'train.npz', 'val.npz', 'scaler.json', 'metadata.json'}:
    raise ValueError('Expected exactly the four declared development input files.')
DATA = PROJECT / '.cache' / 'gpu-notebook' / ATTEMPT / 'inner24'
DATA.mkdir(exist_ok=False)
with zipfile.ZipFile(DATA_ARCHIVE) as archive:
    if len(archive.namelist()) != 4 or set(archive.namelist()) != set(EXPECTED):
        raise ValueError('Transfer archive must contain exactly four files at its root.')
    for name, expected in EXPECTED.items():
        content = archive.read(name)
        if hashlib.sha256(content).hexdigest() != expected:
            raise ValueError(f'Development input hash mismatch: {name}')
        with (DATA / name).open('xb') as handle:
            handle.write(content)
assert {name: sha256(DATA / name) for name in EXPECTED} == EXPECTED
metadata = json.loads((DATA / 'metadata.json').read_text())
print({split: {key: metadata['splits'][split][key] for key in ('meters', 'profiles')}
       for split in ('train', 'val')})

## 5. Declare and run one CUDA pilot

The source plan's `standard_s101` configuration is retained except for the
explicit CUDA backend and 1,000-update budget. Validation uses the same fixed
inner-validation rows, generation seed and physical units. This pilot must not
be pooled with the 6,000-update CPU arms as an equal-budget seed replicate.
Setup and data transfer time are excluded from the reported training-process
wall time; validation is included. Logs and failures remain in the attempt.

In [ ]:
STUDY = PROJECT / 'runs' / 'phase1-development' / f'gpu_pilot_{ATTEMPT}'
STUDY.mkdir(parents=True, exist_ok=False)
CONFIG = dict(PLAN['jobs']['standard_s101'])
CONFIG.update(steps=PILOT_STEPS, device='cuda')
CONFIG_PATH = STUDY / 'config.json'
CONFIG_PATH.write_text(json.dumps(CONFIG, indent=2) + '\n')
RUN = STUDY / 'pilot'
receipt = {
    'phase': 1, 'role': 'post-test development GPU timing pilot',
    'source_commit': SOURCE_COMMIT, 'source_plan_sha256': sha256(PLAN_PATH),
    'provider': PROVIDER, 'runtime': RUNTIME, 'config': CONFIG,
    'data_directory': str(DATA), 'data_hashes': EXPECTED,
    'transfer_archive_sha256': sha256(DATA_ARCHIVE),
    'test_performance_access': False, 'status': 'running',
    'started_at_utc': dt.datetime.now(dt.timezone.utc).isoformat(),
}
(STUDY / 'notebook_receipt.json').write_text(json.dumps(receipt, indent=2) + '\n')
(STUDY / 'pip_freeze.txt').write_text(command([PYTHON, '-m', 'pip', 'freeze'], env=ENV, capture=True) + '\n')
started = time.perf_counter()
returncode = None
try:
    with (STUDY / 'pilot.log').open('x') as log:
        result = subprocess.run([str(PYTHON), '-m', 'fluvius_cgan.development',
                                 '--data', str(DATA), '--run', str(RUN),
                                 '--config', str(CONFIG_PATH)], cwd=PROJECT, env=ENV,
                                text=True, stdout=log, stderr=subprocess.STDOUT)
        returncode = result.returncode
finally:
    receipt.update(status='completed' if returncode == 0 else 'failed_or_interrupted',
                   returncode=returncode, process_wall_seconds=time.perf_counter() - started,
                   finished_at_utc=dt.datetime.now(dt.timezone.utc).isoformat())
    (STUDY / 'notebook_receipt.json').write_text(json.dumps(receipt, indent=2) + '\n')
print((STUDY / 'pilot.log').read_text()[-6000:])
if returncode != 0:
    raise RuntimeError(f'Pilot did not complete; preserve the logs at {STUDY}.')

## 6. Read the timing and development score

Compare **total process wall time for the same workload** when judging a GPU
speedup. This rate includes validation and startup; it is not GPU-only update
throughput. A single seed and an early checkpoint cannot establish convergence,
robust improvement, or an independent validation result.

In [ ]:
manifest = json.loads((RUN / 'manifest.json').read_text())
history = json.loads((RUN / 'history.json').read_text())
summary = {
    'status': manifest['status'],
    'completed_steps': manifest['completed_steps'],
    'process_wall_seconds': round(receipt['process_wall_seconds'], 2),
    'updates_per_second_including_startup_and_validation': round(
        manifest['completed_steps'] / receipt['process_wall_seconds'], 3),
    'best_step': manifest['best_step'],
    'inner_validation_normalized_SW1': manifest['best_validation_sw1'],
    'training_seed': CONFIG['seed'], 'device': CONFIG['device'],
}
print(json.dumps(summary, indent=2))
print('Checkpoint trajectory:')
for row in history:
    print({key: value for key, value in {
        'step': row['step'], 'elapsed_seconds': round(row['elapsed_seconds'], 2),
        'normalized_SW1': row['validation']['normalized_profile_sw1'],
        'diversity_ratio': row['validation']['diversity_ratio'],
    }.items()})

## 7. Optional continuation of a completed segment

This cell is inactive by default. Set `RUN_CONTINUATION=True` before running it
only when extending this pilot to the stated **total** budget. Continuation
requires the same data, configuration, backend and validated runtime context.
It writes a new run directory and preserves the original pilot.

The present continuation path accepts **completed runs at scheduled validation
boundaries**. It does not yet promise recovery from an arbitrary notebook
shutdown or partially written checkpoint. Save completed segments promptly;
export alone does not resume training. CPU and CUDA trajectories are not assumed
to be bitwise identical.

In [ ]:
if RUN_CONTINUATION:
    if type(CONTINUATION_TOTAL_STEPS) is not int or CONTINUATION_TOTAL_STEPS <= PILOT_STEPS:
        raise ValueError('Continuation requires a larger integer total update budget.')
    continuation_config = dict(CONFIG, steps=CONTINUATION_TOTAL_STEPS)
    continuation_config_path = STUDY / 'continuation_config.json'
    with continuation_config_path.open('x') as handle:
        json.dump(continuation_config, handle, indent=2)
    CONTINUED_RUN = STUDY / 'continuation'
    continuation_started = time.perf_counter()
    with (STUDY / 'continuation.log').open('x') as log:
        result = subprocess.run([str(PYTHON), '-m', 'fluvius_cgan.development',
                                 '--data', str(DATA), '--run', str(CONTINUED_RUN),
                                 '--config', str(continuation_config_path),
                                 '--resume', str(RUN / 'last.pt')], cwd=PROJECT, env=ENV,
                                text=True, stdout=log, stderr=subprocess.STDOUT)
    continuation_receipt = {'returncode': result.returncode,
                            'process_wall_seconds': time.perf_counter() - continuation_started,
                            'total_steps': CONTINUATION_TOTAL_STEPS}
    (STUDY / 'continuation_execution.json').write_text(json.dumps(continuation_receipt, indent=2) + '\n')
    print((STUDY / 'continuation.log').read_text()[-6000:])
    if result.returncode != 0:
        raise RuntimeError('Continuation failed; retain both attempts and their logs.')
else:
    print('Optional continuation skipped.')

## 8. Export before the session ends

The archive includes configs, environment versions, provenance, logs, histories
and both selected/latest checkpoints. It excludes the dataset and environment.
Download it promptly. In Colab, use the file browser or the optional download
cell. In Kaggle, save a notebook version with outputs and download the archive
from the Output panel. Keep failed attempts as well as successful ones.

In [ ]:
ARTIFACTS = PROJECT / 'artifacts' / 'phase1-development'
ARTIFACTS.mkdir(parents=True, exist_ok=True)
OUTPUT_ARCHIVE = ARTIFACTS / f'gpu_pilot_{ATTEMPT}.zip'
files = sorted(path for path in STUDY.rglob('*') if path.is_file())
archive_manifest = {str(path.relative_to(STUDY)): sha256(path) for path in files}
with zipfile.ZipFile(OUTPUT_ARCHIVE, 'x', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in files:
        archive.write(path, arcname=str(path.relative_to(STUDY)))
    archive.writestr('archive_manifest.json', json.dumps(archive_manifest, indent=2) + '\n')
with zipfile.ZipFile(OUTPUT_ARCHIVE) as archive:
    for name, expected in archive_manifest.items():
        assert hashlib.sha256(archive.read(name)).hexdigest() == expected, name
print({'archive': str(OUTPUT_ARCHIVE), 'bytes': OUTPUT_ARCHIVE.stat().st_size,
       'sha256': sha256(OUTPUT_ARCHIVE), 'verified_members': len(files)})

In [ ]:
# Optional Colab download. Kaggle users download from the Output panel instead.
DOWNLOAD_IN_COLAB = False
if DOWNLOAD_IN_COLAB:
    from google.colab import files
    files.download(str(OUTPUT_ARCHIVE))

The pilot's measured runtime can guide a free-GPU or Azure decision. GPU hours
needed for the eventual study remain unknown until the chosen configuration,
budget, seed count and validation workload are fixed. Preserve all outcomes and
continue Phase 1; this notebook does not initiate federated learning or Phase 2.